In [32]:
import os

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

INBOX_DIR = "/home/jovyan/data/inbox"

spark = SparkSession.builder.appName("Project1").getOrCreate()


def list_parquet_files(inbox_dir: str):
    if not os.path.isdir(inbox_dir):
        return []

    return sorted(
        os.path.join(inbox_dir, name)
        for name in os.listdir(inbox_dir)
        if name.endswith(".parquet")
    )


parquet_files = list_parquet_files(INBOX_DIR)
print(f"Parquet files to read: {len(parquet_files)}")

if parquet_files:
    df = spark.read.option("mergeSchema", "true").parquet(*parquet_files).withColumn(
        "source_file", F.input_file_name()
    )
else:
    df = None
    print("No parquet files found; skipping.")

Parquet files to read: 2


# Mapping

In [33]:

df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------------------------------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|source_file                                                   |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----

## Zone lookup

In [ ]:
lookup_files = spark.read.option("mergeSchema", "true").parquet(
    f"{INBOX_DIR}/lookup/*.parquet")

zone_lookup = (
    lookup_files
    .filter(F.col("LocationID").isNotNull() & F.col("Zone").isNotNull())
    .select(
        F.col("LocationID").cast("int").alias("LocationID"),
        F.col("Zone").alias("zone_name")
    )
    .dropDuplicates(["LocationID"])
)

## Mapping

In [35]:


# Keep trip rows only
trips = df.filter(
    F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & F.col("PULocationID").isNotNull()
    & F.col("DOLocationID").isNotNull()
)

# Join with trips data
mapped_df = (
    trips.alias("t")
    .join(zone_lookup.alias("pu"), F.col("t.PULocationID") == F.col("pu.LocationID"), "left")
    .join(zone_lookup.alias("do"), F.col("t.DOLocationID") == F.col("do.LocationID"), "left")
    .select(
        F.col("t.tpep_pickup_datetime").alias("pickup_timestamp"),
        F.col("t.tpep_dropoff_datetime").alias("dropoff_timestamp"),
        F.col("t.PULocationID").alias("pickup_location_id"),
        F.col("t.DOLocationID").alias("dropoff_location_id"),
        F.col("pu.zone_name").alias("pickup_zone_name"),
        F.col("do.zone_name").alias("dropoff_zone_name"),
        F.col("t.passenger_count"),
        F.col("t.trip_distance"),
        F.round(
            (
                (
                    F.unix_timestamp(
                        F.col("t.tpep_dropoff_datetime").cast("timestamp"))
                    - F.unix_timestamp(F.col("t.tpep_pickup_datetime").cast("timestamp"))
                ) / 60.0
            ),
            2,
        ).alias("trip_duration_minutes"),
        F.to_date(F.col("t.tpep_pickup_datetime")).alias("pickup_date"),
        F.col("t.source_file"),
        F.current_timestamp().alias("ingested_at"),
    )
)

mapped_df.show(10, truncate=False)

+-------------------+-------------------+------------------+-------------------+-----------------------------+------------------------+---------------+-------------+---------------------+-----------+--------------------------------------------------------------+-------------------------+
|pickup_timestamp   |dropoff_timestamp  |pickup_location_id|dropoff_location_id|pickup_zone_name             |dropoff_zone_name       |passenger_count|trip_distance|trip_duration_minutes|pickup_date|source_file                                                   |ingested_at              |
+-------------------+-------------------+------------------+-------------------+-----------------------------+------------------------+---------------+-------------+---------------------+-----------+--------------------------------------------------------------+-------------------------+
|2025-01-01 00:18:38|2025-01-01 00:26:59|229               |237                |Sutton Place/Turtle Bay North|Upper East Side South  

## Examples of bad rows

